# Phase 12 — Class-Prior and Decision-Shift Analysis

**NetShield: Investigating Dataset-Dependent Behavior in ML-Based Network Intrusion Detection**

### Research question
Phase 11 showed that CORAL substantially reduces covariance mismatch and improves ROC-AUC/PR-AUC, but classification performance remains strongly attack-family dependent. Phase 12 tests whether the remaining cross-dataset gap is partly explained by **class-prior shift and source-side class weighting**, without using CICIDS evaluation labels for model selection or threshold tuning.

### Leakage rule
- CICIDS evaluation labels are **never** used to choose a model, class weight, threshold, or preprocessing setting.
- Thresholds are selected only on the UNSW source-validation split.
- CICIDS labels are used only after all choices are frozen, for final evaluation and diagnostic analysis.

### Experiments
1. Baseline semantic RF.
2. Source class-weighted semantic RF.
3. CORAL RF.
4. CORAL + source class weighting.
5. Source-only threshold selection for each model.
6. Final CICIDS evaluation.
7. Target-label-free score/prior diagnostics.
8. Attack-family analysis after the frozen comparison.

The purpose is **not** to search for the best classifier. It is to determine whether class-prior/decision shift explains part of the transfer gap.


In [ ]:
from pathlib import Path
import json, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_UNSW = PROJECT_ROOT / "data" / "raw" / "UNSW-NB15"
RAW_CICIDS = PROJECT_ROOT / "data" / "raw" / "CICIDS2017"
OUT = PROJECT_ROOT / "outputs" / "class_prior_analysis"
TABLE_DIR = OUT / "tables"
PLOT_DIR = OUT / "plots"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output folder:", OUT)


## 1. Load the source data

This cell uses the official UNSW training set and testing set. The model features are the same 11 semantically aligned features used in Phase 11.


In [ ]:
COMMON = [
    "flow_duration_sec", "fwd_packets", "bwd_packets", "fwd_bytes", "bwd_bytes",
    "total_packets", "total_bytes", "packet_rate", "fwd_packet_rate",
    "bwd_packet_rate", "avg_packet_size"
]

def find_csv(folder, name):
    matches = list(folder.rglob(name))
    if not matches:
        raise FileNotFoundError(f"Could not find {name} under {folder}")
    return matches[0]

unsw_train_path = find_csv(RAW_UNSW, "UNSW_NB15_training-set.csv")
unsw_test_path = find_csv(RAW_UNSW, "UNSW_NB15_testing-set.csv")

unsw_train_raw = pd.read_csv(unsw_train_path, low_memory=False)
unsw_test_raw = pd.read_csv(unsw_test_path, low_memory=False)
print("UNSW train:", unsw_train_raw.shape)
print("UNSW test :", unsw_test_raw.shape)


In [ ]:
def build_unsw_common(df):
    out = pd.DataFrame(index=df.index)
    dur = pd.to_numeric(df["dur"], errors="coerce")
    fp = pd.to_numeric(df["spkts"], errors="coerce")
    bp = pd.to_numeric(df["dpkts"], errors="coerce")
    fb = pd.to_numeric(df["sbytes"], errors="coerce")
    bb = pd.to_numeric(df["dbytes"], errors="coerce")
    out["flow_duration_sec"] = dur
    out["fwd_packets"] = fp
    out["bwd_packets"] = bp
    out["fwd_bytes"] = fb
    out["bwd_bytes"] = bb
    out["total_packets"] = fp + bp
    out["total_bytes"] = fb + bb
    safe_dur = dur.replace(0, np.nan)
    safe_tp = (fp + bp).replace(0, np.nan)
    out["packet_rate"] = (fp + bp) / safe_dur
    out["fwd_packet_rate"] = fp / safe_dur
    out["bwd_packet_rate"] = bp / safe_dur
    out["avg_packet_size"] = (fb + bb) / safe_tp
    return out[COMMON]

X_unsw = build_unsw_common(unsw_train_raw)
y_unsw = pd.to_numeric(unsw_train_raw["label"], errors="coerce").astype("int8")

X_unsw_test = build_unsw_common(unsw_test_raw)
y_unsw_test = pd.to_numeric(unsw_test_raw["label"], errors="coerce").astype("int8")

print("Source features:", X_unsw.shape)
print("Source attack prior:", y_unsw.mean())


## 2. Load CICIDS safely in chunks

Only the 11 semantic inputs and the label are loaded. `source_file` is retained only for later diagnostics and is never a model feature.


In [ ]:
NEEDED = [
    'Flow Duration','Total Fwd Packets','Total Backward Packets',
    'Total Length of Fwd Packets','Total Length of Bwd Packets',
    'Flow Packets/s','Fwd Packets/s','Bwd Packets/s',
    'Average Packet Size','Label'
]
frames = []
for p in sorted(RAW_CICIDS.rglob('*.csv')):
    h = pd.read_csv(p, nrows=0)
    lookup = {str(x).strip().replace('\ufeff','').lower(): x for x in h.columns}
    sel = {x: lookup[x.lower()] for x in NEEDED if x.lower() in lookup}
    if 'Label' not in sel:
        print('skip', p.name)
        continue
    chunks=[]
    for chunk in pd.read_csv(p, usecols=list(sel.values()), chunksize=100_000, low_memory=True):
        chunk = chunk.rename(columns={v:k for k,v in sel.items()})
        for c in NEEDED:
            if c != 'Label' and c in chunk.columns:
                chunk[c] = pd.to_numeric(chunk[c], errors='coerce')
        chunk['Label'] = chunk['Label'].astype(str).str.strip()
        chunk['source_file'] = p.name
        chunks.append(chunk)
    z = pd.concat(chunks, ignore_index=True)
    frames.append(z)
    print('loaded', p.name, len(z))
    del chunks, z
    gc.collect()

cicids = pd.concat(frames, ignore_index=True)
cicids['source_file'] = cicids['source_file'].astype('category')
print('CICIDS:', cicids.shape)


In [ ]:
def build_cicids_common(df):
    out = pd.DataFrame(index=df.index)
    dur = pd.to_numeric(df['Flow Duration'], errors='coerce') / 1e6
    fp = pd.to_numeric(df['Total Fwd Packets'], errors='coerce')
    bp = pd.to_numeric(df['Total Backward Packets'], errors='coerce')
    fb = pd.to_numeric(df['Total Length of Fwd Packets'], errors='coerce')
    bb = pd.to_numeric(df['Total Length of Bwd Packets'], errors='coerce')
    out['flow_duration_sec'] = dur
    out['fwd_packets'] = fp
    out['bwd_packets'] = bp
    out['fwd_bytes'] = fb
    out['bwd_bytes'] = bb
    out['total_packets'] = fp + bp
    out['total_bytes'] = fb + bb
    out['packet_rate'] = pd.to_numeric(df['Flow Packets/s'], errors='coerce')
    out['fwd_packet_rate'] = pd.to_numeric(df['Fwd Packets/s'], errors='coerce')
    out['bwd_packet_rate'] = pd.to_numeric(df['Bwd Packets/s'], errors='coerce')
    out['avg_packet_size'] = pd.to_numeric(df['Average Packet Size'], errors='coerce')
    return out[COMMON]

X_cic = build_cicids_common(cicids)
y_cic = cicids['Label'].astype(str).str.upper().ne('BENIGN').astype('int8')

print('CICIDS attack prior:', y_cic.mean())
print('CICIDS label counts:')
print(y_cic.value_counts())


## 3. Leakage-safe source/evaluation splits

The source split is used to select thresholds. CICIDS is split into an unlabeled adaptation pool and a final evaluation set. The evaluation labels remain untouched until the final scoring cells.


In [ ]:
src_fit_idx, src_val_idx = train_test_split(
    np.arange(len(X_unsw)), test_size=0.20, stratify=y_unsw, random_state=RANDOM_STATE
)

cic_pool_idx, cic_eval_idx = train_test_split(
    np.arange(len(X_cic)), test_size=0.20, stratify=y_cic, random_state=RANDOM_STATE
)

rng = np.random.default_rng(RANDOM_STATE)
ADAPT_MAX = min(300_000, len(cic_pool_idx))
cic_adapt_idx = rng.choice(cic_pool_idx, size=ADAPT_MAX, replace=False)

print('UNSW source-fit:', len(src_fit_idx))
print('UNSW source-val:', len(src_val_idx))
print('CICIDS adaptation pool sample:', len(cic_adapt_idx))
print('CICIDS evaluation:', len(cic_eval_idx))


In [ ]:
def clean(X):
    X = np.asarray(X, dtype=np.float64).copy()
    X[~np.isfinite(X)] = np.nan
    X[np.abs(X) > np.finfo(np.float32).max] = np.nan
    return X

def fit_preprocessor(X):
    imp = SimpleImputer(strategy='median')
    Xi = imp.fit_transform(clean(X))
    sc = StandardScaler()
    return imp, sc, sc.fit_transform(Xi).astype('float64')

def transform(X, imp, sc):
    return sc.transform(imp.transform(clean(X))).astype('float64')

def cov(Z):
    return np.atleast_2d(np.cov(np.asarray(Z, dtype=np.float64), rowvar=False))

def sym_sqrt(M, eps=1e-6):
    M=(M+M.T)/2
    w,V=np.linalg.eigh(M); w=np.clip(w,eps,None)
    return (V*np.sqrt(w))@V.T

def sym_inv_sqrt(M, eps=1e-6):
    M=(M+M.T)/2
    w,V=np.linalg.eigh(M); w=np.clip(w,eps,None)
    return (V*(1/np.sqrt(w)))@V.T

def fit_coral(Xs, Xt):
    imp,sc,Zs=fit_preprocessor(Xs)
    Zt=transform(Xt,imp,sc)
    Cs,Ct=cov(Zs),cov(Zt)
    A=sym_inv_sqrt(Cs)@sym_sqrt(Ct)
    mu_s,mu_t=Zs.mean(0),Zt.mean(0)
    Zs_aligned=(Zs-mu_s)@A+mu_t
    before=np.linalg.norm(Cs-Ct,ord='fro')
    after=np.linalg.norm(cov(Zs_aligned)-Ct,ord='fro')
    if not np.isfinite(after) or after >= before:
        raise RuntimeError(f'CORAL covariance alignment failed: {before} -> {after}')
    return {'imp':imp,'sc':sc,'A':A,'mu_s':mu_s,'mu_t':mu_t,'before':before,'after':after}

def source_coral_transform(X,c):
    Z=transform(X,c['imp'],c['sc'])
    return ((Z-c['mu_s'])@c['A']+c['mu_t']).astype('float64')

def target_coral_transform(X,c):
    return transform(X,c['imp'],c['sc'])


## 4. Prepare baseline and CORAL representations

The target adaptation sample is unlabeled. It can influence CORAL's covariance alignment, but not class labels or threshold selection.


In [ ]:
Xsrc_fit = X_unsw.iloc[src_fit_idx]
y_src_fit = y_unsw.iloc[src_fit_idx].to_numpy()
Xsrc_val = X_unsw.iloc[src_val_idx]
y_src_val = y_unsw.iloc[src_val_idx].to_numpy()
Xcic_eval = X_cic.iloc[cic_eval_idx]
y_cic_eval = y_cic.iloc[cic_eval_idx].to_numpy()
Xcic_adapt = X_cic.iloc[cic_adapt_idx]

base_imp = SimpleImputer(strategy='median')
Xsrc_base = base_imp.fit_transform(clean(Xsrc_fit))
Xval_base = base_imp.transform(clean(Xsrc_val))
Xeval_base = base_imp.transform(clean(Xcic_eval))

coral = fit_coral(Xsrc_fit, Xcic_adapt)
Xsrc_coral = source_coral_transform(Xsrc_fit, coral)
Xval_coral = source_coral_transform(Xsrc_val, coral)
Xeval_coral = target_coral_transform(Xcic_eval, coral)

print('CORAL covariance distance:', coral['before'], '->', coral['after'])


## 5. Train the four controlled models

The only intervention is **source-side class weighting**. We are deliberately not changing the algorithm, number of trees, or semantic features. This isolates whether class-prior imbalance contributes to transfer behavior.


In [ ]:
rf_params = dict(n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1, max_features='sqrt', min_samples_leaf=2)

models = {
    'baseline_unweighted': (RandomForestClassifier(**rf_params), Xsrc_base, Xval_base, Xeval_base),
    'baseline_balanced': (RandomForestClassifier(**rf_params, class_weight='balanced'), Xsrc_base, Xval_base, Xeval_base),
    'coral_unweighted': (RandomForestClassifier(**rf_params), Xsrc_coral, Xval_coral, Xeval_coral),
    'coral_balanced': (RandomForestClassifier(**rf_params, class_weight='balanced'), Xsrc_coral, Xval_coral, Xeval_coral),
}

trained = {}
for name,(model,Xfit,Xval,Xeval) in models.items():
    model.fit(Xfit, y_src_fit)
    trained[name] = {
        'model': model,
        'source_val_prob': model.predict_proba(Xval)[:,1],
        'target_prob': model.predict_proba(Xeval)[:,1]
    }
    print('trained', name)


## 6. Select every threshold using UNSW source validation only

Each model gets its own threshold. CICIDS labels are not used here.


In [ ]:
threshold_grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
threshold_rows=[]
selected={}
for name,obj in trained.items():
    rows=[]
    for t in threshold_grid:
        pred=(obj['source_val_prob']>=t).astype('int8')
        rows.append({
            'model':name,'threshold':float(t),
            'precision':precision_score(y_src_val,pred,zero_division=0),
            'recall':recall_score(y_src_val,pred,zero_division=0),
            'f1':f1_score(y_src_val,pred,zero_division=0)
        })
    df=pd.DataFrame(rows)
    threshold_rows.append(df)
    selected[name]=float(df.loc[df['f1'].idxmax(),'threshold'])

thresholds=pd.concat(threshold_rows,ignore_index=True)
thresholds.to_csv(TABLE_DIR/'01_source_threshold_selection.csv',index=False)
selected_df=pd.DataFrame({'model':list(selected),'selected_threshold':[selected[k] for k in selected]})
selected_df.to_csv(TABLE_DIR/'02_selected_thresholds.csv',index=False)
print(selected_df.to_string(index=False))


In [ ]:
for name in trained:
    d=thresholds[thresholds.model==name]
    plt.figure(figsize=(8,5))
    plt.plot(d.threshold,d.f1)
    plt.axvline(selected[name], linestyle='--', label=f"selected={selected[name]:.2f}")
    plt.xlabel('Threshold'); plt.ylabel('Source-validation F1'); plt.title(f'Source threshold selection: {name}')
    plt.legend(); plt.tight_layout(); plt.savefig(PLOT_DIR/f'03_threshold_{name}.png',dpi=160); plt.show()


## 7. Final frozen CICIDS evaluation

Only now are CICIDS evaluation labels used. These results are the fair comparison of the four interventions.


In [ ]:
def evaluate(y,prob,t):
    pred=(prob>=t).astype('int8')
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {
        'precision':precision_score(y,pred,zero_division=0),
        'recall':recall_score(y,pred,zero_division=0),
        'f1':f1_score(y,pred,zero_division=0),
        'macro_f1':f1_score(y,pred,average='macro',zero_division=0),
        'pr_auc':average_precision_score(y,prob),
        'roc_auc':roc_auc_score(y,prob),
        'fpr':fp/(fp+tn) if (fp+tn) else 0,
        'fnr':fn/(fn+tp) if (fn+tp) else 0,
        'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp),
        'predicted_attack_rate':float(pred.mean()),
        'threshold':float(t)
    }

results=[]
for name,obj in trained.items():
    r=evaluate(y_cic_eval,obj['target_prob'],selected[name])
    r['model']=name
    results.append(r)

results_df=pd.DataFrame(results)[['model','threshold','precision','recall','f1','macro_f1','pr_auc','roc_auc','fpr','fnr','predicted_attack_rate','tn','fp','fn','tp']]
results_df.to_csv(TABLE_DIR/'04_frozen_cicids_class_prior_results.csv',index=False)
display(results_df)


## 8. Compare the class-weighting intervention directly

This isolates whether balancing the source training objective improves transfer.


In [ ]:
comparison=[]
for base,bal in [('baseline_unweighted','baseline_balanced'),('coral_unweighted','coral_balanced')]:
    a=results_df[results_df.model==base].iloc[0]
    b=results_df[results_df.model==bal].iloc[0]
    comparison.append({
        'pair':f'{base} -> {bal}',
        'delta_f1':b.f1-a.f1,
        'delta_macro_f1':b.macro_f1-a.macro_f1,
        'delta_recall':b.recall-a.recall,
        'delta_precision':b.precision-a.precision,
        'delta_fpr':b.fpr-a.fpr,
        'delta_pr_auc':b.pr_auc-a.pr_auc,
        'delta_roc_auc':b.roc_auc-a.roc_auc
    })
comparison_df=pd.DataFrame(comparison)
comparison_df.to_csv(TABLE_DIR/'05_class_weight_effect.csv',index=False)
display(comparison_df)


## 9. Target-label-free decision diagnostics

The actual CICIDS attack prior is **not** used to tune anything. We instead inspect the predicted-positive rate and score distribution using target features only. This helps distinguish a probability/decision shift from a representation-ranking problem.


In [ ]:
diag=[]
for name,obj in trained.items():
    p=obj['target_prob']
    diag.append({
        'model':name,
        'mean_score':float(np.mean(p)),
        'median_score':float(np.median(p)),
        'p90_score':float(np.quantile(p,.90)),
        'p99_score':float(np.quantile(p,.99)),
        'predicted_attack_rate_at_source_threshold':float(np.mean(p>=selected[name])),
        'source_threshold':selected[name]
    })
diag_df=pd.DataFrame(diag)
diag_df.to_csv(TABLE_DIR/'06_target_score_shift_diagnostic.csv',index=False)
display(diag_df)


In [ ]:
plt.figure(figsize=(9,5))
for name,obj in trained.items():
    plt.hist(obj['target_prob'],bins=50,alpha=.35,label=name)
plt.xlabel('Predicted attack probability'); plt.ylabel('Count'); plt.title('CICIDS target score distributions')
plt.legend(); plt.tight_layout(); plt.savefig(PLOT_DIR/'07_target_score_distributions.png',dpi=160); plt.show()


## 10. Attack-family analysis — diagnostic only

This cell uses CICIDS evaluation labels **after the model/threshold choices are frozen**. Tiny attack families should not be treated as strong evidence.


In [ ]:
labels_eval=cicids.iloc[cic_eval_idx]['Label'].astype(str).str.strip().to_numpy()
family_rows=[]
for label in pd.Series(labels_eval).value_counts().index:
    mask=(labels_eval==label)
    row={'attack_label':label,'n':int(mask.sum())}
    for name,obj in trained.items():
        pred=(obj['target_prob']>=selected[name]).astype('int8')
        row[f'{name}_detection_rate']=float(pred[mask].mean())
    family_rows.append(row)
family_df=pd.DataFrame(family_rows).sort_values('n',ascending=False)
family_df.to_csv(TABLE_DIR/'08_attack_family_effect_class_prior.csv',index=False)
display(family_df)


## 11. Controlled interpretation

Use these rules when interpreting Phase 12:

- **Class weighting helps both recall/F1 and does not create an unacceptable FPR increase:** class-prior imbalance is a meaningful contributor to transfer failure.
- **Class weighting helps CORAL but not the baseline:** the interaction between domain alignment and class imbalance is important.
- **Class weighting barely changes results while CORAL ranking remains better:** the remaining problem is less likely to be simple class-prior imbalance and more likely to be attack-family/representation mismatch.
- **Source-selected thresholds transfer poorly despite class weighting:** decision calibration remains dataset dependent.
- **Attack-family effects remain large:** global covariance alignment is insufficient to make all attack types semantically equivalent across datasets.

Do **not** claim that a target threshold is optimal from the target diagnostic. It is diagnostic evidence only.


In [ ]:
evidence={
    'source_attack_prior':float(y_unsw.mean()),
    'cicids_attack_prior_diagnostic_only':float(y_cic.mean()),
    'coral_covariance_distance_before':float(coral['before']),
    'coral_covariance_distance_after':float(coral['after']),
    'selected_source_thresholds':selected,
    'frozen_results':results_df.to_dict(orient='records'),
    'class_weight_effect':comparison_df.to_dict(orient='records')
}
with open(TABLE_DIR/'09_phase12_evidence.json','w') as f:
    json.dump(evidence,f,indent=2)

print('Phase 12 complete.')
print('Results saved to:', TABLE_DIR)
